In [ ]:
from google.colab import files

uploaded = files.upload()  # A window will open to upload files

Saving mixtec-train.txt to mixtec-train.txt
Saving mixtec-val.txt to mixtec-val.txt
Saving spanish-train.txt to spanish-train.txt
Saving spanish-val.txt to spanish-val.txt


In [ ]:
# 📦 Install the necessary libraries: datasets, transformers, and huggingface_hub
!pip install datasets transformers huggingface_hub --upgrade

import pandas as pd  # Library for handling data structures like DataFrames
import json  # Library for working with JSON data
from datasets import load_dataset, DatasetDict  # For loading and handling datasets
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSeq2SeqLM  # Libraries for working with transformers
import wandb  # For monitoring and saving experiments in Weights & Biases
import torch  # For working with PyTorch and devices like GPU

# 📂 Read the training files (in Mixteco and Spanish)
with open("mixtec-train.txt", "r", encoding="utf-8") as f_mix_train, open("spanish-train.txt", "r", encoding="utf-8") as f_esp_train:
    mixteco_train_sentences = f_mix_train.readlines()  # Read Mixteco sentences
    espanol_train_sentences = f_esp_train.readlines()  # Read Spanish sentences

# 📌 Ensure that the training files have the same number of lines (data consistency)
assert len(mixteco_train_sentences) == len(espanol_train_sentences), "Training files do not have the same number of lines."

# 📂 Read the validation files
with open("mixtec-val.txt", "r", encoding="utf-8") as f_mix_val, open("spanish-val.txt", "r", encoding="utf-8") as f_esp_val:
    mixteco_val_sentences = f_mix_val.readlines()  # Mixteco sentences for validation
    espanol_val_sentences = f_esp_val.readlines()  # Spanish sentences for validation

# 📌 Ensure that the validation files have the same number of lines
assert len(mixteco_val_sentences) == len(espanol_val_sentences), "Validation files do not have the same number of lines."

# 📊 Create DataFrames with training and validation sentences
train_df = pd.DataFrame({
    "mixtec": [line.strip() for line in mixteco_train_sentences],  # Mixteco sentences as rows
    "spanish": [line.strip() for line in espanol_train_sentences]  # Spanish sentences as rows
})

val_df = pd.DataFrame({
    "mixtec": [line.strip() for line in mixteco_val_sentences],  # Mixteco sentences for validation
    "spanish": [line.strip() for line in espanol_val_sentences]  # Spanish sentences for validation
})

# 💾 Save the DataFrames as JSON files for later loading
train_dataset_path = "train_mixteco_espanol.json"
val_dataset_path = "val_mixteco_espanol.json"

# Convert DataFrames to JSON format
train_df.to_json(train_dataset_path, orient="records", force_ascii=False, indent=4)
val_df.to_json(val_dataset_path, orient="records", force_ascii=False, indent=4)

print(f"Datasets saved in {train_dataset_path} and {val_dataset_path}")  # Confirmation of saving

# 📥 Load the datasets from the JSON files
train_dataset = load_dataset("json", data_files=train_dataset_path)["train"]  # Load training dataset
val_dataset = load_dataset("json", data_files=val_dataset_path)["train"]  # Load validation dataset

# Create a DatasetDict grouping the training and validation datasets
dataset = DatasetDict({
    "train": train_dataset,
    "test": val_dataset
})

# ⚙️ Load the tokenizer and the M2M100 translation model from Facebook
model_name = "facebook/m2m100_418M"  # Pretrained model for multilingual translation
tokenizer = AutoTokenizer.from_pretrained(model_name)  # Load the tokenizer for the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)  # Load the translation model

# 🔄 Preprocessing function to tokenize the data
def preprocess_function(examples):
    inputs = [f"{text} </s>" for text in examples["mixtec"]]  # Input is in Mixteco
    targets = [f"{text} </s>" for text in examples["spanish"]]  # Expected output is in Spanish

    # Tokenize the inputs and outputs
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=128)
    labels = tokenizer(targets, padding="max_length", truncation=True, max_length=128)

    model_inputs["labels"] = labels["input_ids"]  # Assign label IDs for training
    return model_inputs

# 🔄 Tokenize the dataset using the preprocessing function
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 🖥️ Set up the device for training (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # Print whether GPU or CPU is being used
model.to(device)  # Move the model to the appropriate device

# 📌 Set up the training parameters
training_args = TrainingArguments(
    output_dir="./results",  # Output directory for results
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",  # Save the model at the end of each epoch
    per_device_train_batch_size=8,  # Batch size for training (adjust based on available memory)
    per_device_eval_batch_size=8,  # Batch size for evaluation
    num_train_epochs=5,  # Number of training epochs
    logging_dir="./logs",  # Directory for logs
    logging_steps=50,  # Log frequency
    learning_rate=3e-5,  # Learning rate
    weight_decay=0.01,  # Regularization to avoid overfitting
    warmup_ratio=0.06,  # 6% of the training for warmup
    fp16=True,  # Enable FP16 if using GPU
    report_to="wandb",  # Report metrics to Weights & Biases
    push_to_hub=False  # Do not push the model to Hugging Face Hub
)

# Connect to Weights & Biases for experiment tracking
wandb.login()
wandb.init(project="m2m100_finetuning_mixteco_espanol", name="m2m100_train_run")

# 📚 Create the Trainer to handle training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],  # Training dataset
    eval_dataset=tokenized_datasets["test"],  # Validation dataset
    tokenizer=tokenizer,  # Tokenizer used for inputs
)

# 🚀 Start training the model
trainer.train()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.28.1
    Uninstalling huggingface-hub-0.28.1:
      Successfully uninstalled huggingface-hub-0.28.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.3
    Uninstalling transformers-4.48.3:
      Successfully uninstalled transformers-4.48.3
Datasets guardados en train_mixteco_espanol.json y val_mixteco_espa

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


<ipython-input-2-ed406d337033>:99: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
1,0.615100,0.163359
2,0.506700,0.158099
3,0.475400,0.158520
4,0.409500,0.157870
5,0.369200,0.158550


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2810: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=7295, training_loss=0.7060915699227368, metrics={'train_runtime': 1167.4472, 'train_samples_per_second': 49.977, 'train_steps_per_second': 6.249, 'total_flos': 1.580496475127808e+16, 'train_loss': 0.7060915699227368, 'epoch': 5.0})

In [ ]:
# 📦 Install the necessary libraries: evaluate and sacrebleu
!pip install evaluate sacrebleu

import torch  # Library for working with PyTorch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM  # Libraries from Hugging Face for working with transformers
import evaluate  # Library for evaluation metrics

# 🔽 Load the fine-tuned model
model_name = "./m2m100_modelo_mixteco_espanol"  # Path where the fine-tuned model is saved
tokenizer = AutoTokenizer.from_pretrained(model_name)  # Load the tokenizer associated with the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")  # Load the model and move it to GPU if available

# 📦 Set the model to evaluation mode (disables dropout layers and other training-specific operations)
model.eval()

# 📂 Load validation data (Mixteco and Spanish sentences)
with open("mixtec-val.txt", "r", encoding="utf-8") as f_mix, open("spanish-val.txt", "r", encoding="utf-8") as f_esp:
    mixteco_sentences = [line.strip() for line in f_mix.readlines()]  # Read Mixteco sentences
    spanish_references = [line.strip() for line in f_esp.readlines()]  # Read Spanish reference translations

# 📌 Ensure that the number of lines in both files is equal (data consistency check)
assert len(mixteco_sentences) == len(spanish_references), "The files do not have the same number of lines."

# 🔄 Function to translate texts in batches
def translate_texts(texts, model, tokenizer, batch_size=4, max_length=128):
    translated_sentences = []  # List to store translated sentences

    # Process texts in batches
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]  # Get a small batch of texts
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(model.device)  # Tokenize the batch

        with torch.no_grad():  # Disable gradient calculation for inference
            outputs = model.generate(**inputs, max_length=max_length)  # Generate translations

        decoded_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)  # Decode the translations into text
        translated_sentences.extend(decoded_texts)  # Add the translations to the list

    return translated_sentences  # Return all translated sentences

# 🔽 Get predictions (translations) in batches
translated_sentences = translate_texts(mixteco_sentences, model, tokenizer, batch_size=4)

# ⚖️ Evaluate the translations using BLEU and TER scores
bleu = evaluate.load("bleu")  # Load the BLEU metric
ter = evaluate.load("ter")  # Load the TER (Translation Edit Rate) metric

# Compute BLEU score
bleu_score = bleu.compute(predictions=translated_sentences, references=[[ref] for ref in spanish_references])

# Compute TER score
ter_score = ter.compute(predictions=translated_sentences, references=[[ref] for ref in spanish_references])

# 📊 Print evaluation results (BLEU and TER)
print(f"BLEU Score: {bleu_score['bleu']:.4f}")  # Print the BLEU score
print(f"TER Score: {ter_score['score']:.4f}")  # Print the TER score (adjust the key according to the correct value)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.8 MB/s eta 0:00:00


BLEU Score: 0.0511


KeyError: 'ter'

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./m2m100_modelo_mixteco_espanol")  # Save the model's weights and configuration to a specified directory
tokenizer.save_pretrained("./m2m100_modelo_mixteco_espanol")  # Save the tokenizer configuration and vocab to the same directory

# Print a message confirming that the training is complete and the model has been saved
print("Entrenamiento completado y modelo guardado.")


Entrenamiento completado y modelo guardado.
